# Checkpoint-3500 Full Test Inference

`checkpoint-3500`을 로드해서 test 전체를 full-order 단일 호출로 추론하고 제출 CSV를 저장합니다.


In [ ]:
# 1) Install dependencies, then restart runtime once
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_checkpoint3500_submit_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "jedi",
        "pandas==2.2.2",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")

In [ ]:
# 2) Setup + paths
from google.colab import drive
drive.mount("/content/drive")

import gc
import json
import os
import random
import re
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import PeftModel

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

RUN_ID = "20260712_234828"
CHECKPOINT_NAME = "checkpoint-3500"
LGT_ROOT_CANDIDATES = [
    "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_multitask_v1",
    "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_direct_order_multitask_v1",
]

SEED = 42
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
TEST_INFERENCE_ROWS = None  # None = full test

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR

def resolve_output_dir(run_id):
    checked = []
    for root in LGT_ROOT_CANDIDATES:
        candidate = os.path.join(root, "runs", run_id, "lgt_multitask")
        checked.append(candidate)
        if os.path.isdir(candidate):
            return candidate
    raise RuntimeError("Could not find run output. Checked:\n" + "\n".join(checked))

OUTPUT_DIR = resolve_output_dir(RUN_ID)
ADAPTER_DIR = os.path.join(OUTPUT_DIR, CHECKPOINT_NAME)
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")), ADAPTER_DIR

SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission_checkpoint3500.csv")
DEBUG_PATH = os.path.join(OUTPUT_DIR, "submission_checkpoint3500_debug.csv")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("adapter:", ADAPTER_DIR)
print("submission:", SUBMIT_PATH)

In [ ]:
# 3) Test dataset + prompt helpers
def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()

def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]

def parse_order_prediction(text):
    match = re.fullmatch(
        r"\s*\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]\s*",
        str(text),
    )
    if not match:
        return None
    values = [int(value) for value in match.groups()]
    return values if sorted(values) == [1, 2, 3, 4] else None

def task_instruction(sentence):
    return (
        f"Caption:\n{sentence}\n\n"
        "Question: Arrange all images in chronological order.\n"
        "Return only one Python-style list of image numbers, such as [1, 2, 3, 4].\n"
        "Do not output any explanation."
    )

def make_order_messages(example):
    content = []
    for input_number in range(1, 5):
        content.append({"type": "text", "text": f"\nImage {input_number}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + example["instruction"]})
    return [{"role": "user", "content": content}]

class TestOrderDataset:
    def __init__(self, dataframe, image_root):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_root = image_root

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        sample_id = str(row["Id"])
        sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
        return {
            "Id": sample_id,
            "image_paths": row_image_paths(row, self.image_root),
            "instruction": task_instruction(sentence),
        }

test_df = pd.read_csv(TEST_CSV)
test_df["Id"] = test_df["Id"].astype(str)
if TEST_INFERENCE_ROWS is not None:
    test_df = test_df.iloc[:TEST_INFERENCE_ROWS].copy().reset_index(drop=True)

test_dataset = TestOrderDataset(test_df, TEST_IMAGE_DIR)
print("test rows:", len(test_dataset))
display(test_df.head())

In [ ]:
# 4) Load checkpoint-3500 model
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def disable_sampling_warnings(model):
    generation_config = getattr(model, "generation_config", None)
    if generation_config is None:
        return
    generation_config.do_sample = False
    generation_config.temperature = None
    generation_config.top_p = None
    generation_config.top_k = None

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
model.config.use_cache = True
disable_sampling_warnings(model)

print("loaded:", ADAPTER_DIR)

In [ ]:
# 5) Full test inference + save CSV
@torch.no_grad()
def generate_order(example, max_new_tokens=24):
    text = processor.apply_chat_template(make_order_messages(example), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    inputs = processor(text=[text], images=images, return_tensors="pt").to(model.device)
    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    output_ids = generated_ids[0, inputs.input_ids.shape[1]:]
    return processor.decode(output_ids, skip_special_tokens=True).strip()

rows = []
invalid_count = 0

for index in tqdm(range(len(test_dataset)), desc="checkpoint-3500 test inference"):
    example = test_dataset[index]
    raw_output = generate_order(example)
    prediction = parse_order_prediction(raw_output)
    valid_output = prediction is not None
    if prediction is None:
        invalid_count += 1
        prediction = [1, 2, 3, 4]
    rows.append({
        "Id": example["Id"],
        "Answer": str(prediction),
        "raw_output": raw_output,
        "valid_output": valid_output,
    })

debug_df = pd.DataFrame(rows)
submission_df = debug_df[["Id", "Answer"]].copy()

submission_df.to_csv(SUBMIT_PATH, index=False)
debug_df.to_csv(DEBUG_PATH, index=False)

print("invalid outputs:", invalid_count, "/", len(debug_df))
print("saved submission:", SUBMIT_PATH)
print("saved debug:", DEBUG_PATH)
display(submission_df.head())

In [ ]:
# 6) Cleanup
del model
del base_model
gc.collect()
torch.cuda.empty_cache()
print("done")